# Notebook: Obesity Risk Classification
Romain Sebire - 125 009 460

**Objective:** Classify individuals into 7 obesity risk categories based on lifestyle and physical attributes, as part of a Kaggle competition.

**Dataset:** Multi-class classification dataset with 17 features (Age, Height, Weight, eating habits, physical activity, etc.) and 7 target classes from Insufficient_Weight to Obesity_Type_III.

**Approach:** Comparative study of multiple classifiers (Logistic Regression, KNN, SVM, Random Forest, XGBoost), feature engineering with BMI, and hyperparameter tuning.

**Key Result:** Achieved **0.91257** public score and **0.90715** private score on Kaggle using XGBoost with BMI feature engineering.

## Table of Contents

1. [Data Loading and Verification](#Data-Loading-and-Verification)
2. [Exploratory Data Analysis (EDA)](#Exploratory-Data-Analysis-(EDA))
    - [Training Data Visualization](#Training-Data-Visualization)
    - [Test Data Visualization](#Test-Data-Visualization)
    - [Target Variable Relationships](#Visualization-of-Relationships-with-the-Target-Variable-(NObeyesdad))
3. [BMI Feature Engineering](#Adding-a-New-Explanatory-Variable-(BMI))
4. [Model Training](#Model-Training)
    - [Preprocessing](#Preprocessing:-Encoding-and-Normalization)
    - [Model Comparison](#Train-Test-Split)
    - [Hyperparameter Tuning](#Hyperparameter-Search-to-Improve-XGBoost-Performance)
5. [Final Submission](#Training-with-100%-of-Training-Data-and-Submission-File-Creation)
6. [Kaggle Results](#Kaggle-Results)

## Data Loading and Verification

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

# Exibição das primeiras linhas
df_train.head()

In [ ]:
# Statistics on numerical data
df_train.describe(include='all')

In [ ]:
# Check for missing values
missing_values_train = df_train.isnull().sum()
print("Missing values in train dataset:\n", missing_values_train)

missing_values_test = df_test.isnull().sum()
print("Missing values in test dataset:\n", missing_values_test)

No missing values in either the training or test data.

In [ ]:
df_train.info()

### Notes on the Explanatory Variables

Numerical data: Age, Height, Weight, FCVC, NCP, CH2O, FAF, TUE

Categorical data: Gender, family history of overweight, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS

#### Meaning of Each Variable

id — Unique row identifier

Gender — Sex of the individual (Male or Female)

Age — Age in years

Height — Height in meters

Weight — Weight in kilograms

family\_history\_with\_overweight — Family history of overweight? (yes or no)

FAVC — Frequent consumption of high-calorie food (yes or no)

FCVC — Frequency of vegetable consumption (on a scale, generally 1 to 3)

NCP — Number of main meals per day (generally 1 to 4)

CAEC — Food consumption between meals: no, Sometimes, Frequently, Always

SMOKE — Smoker? (yes or no)

CH2O — Liters of water consumed per day (generally between 0 and 3)

SCC — Monitors caloric intake? (yes or no)

FAF — Physical activity frequency (from 0 = never to 3 = daily)

TUE — Time using technology per day (in hours)

CALC — Alcohol consumption: no, Sometimes, Frequently, Always

MTRANS — Main mode of transportation: Public\_Transportation, Walking, Automobile, etc.

NObeyesdad — Target: body mass class (e.g., Normal\_Weight, Obesity\_Type\_I, etc.)

## Exploratory Data Analysis (EDA)

### Training Data Visualization

In [ ]:
df = df_train.copy()

# Target analysis
df['NObeyesdad'].value_counts().plot(kind='bar')

# Visualization of numerical variable distributions via histogram
numeric_columns = df.select_dtypes(include=['float64']).columns

plt.figure(figsize=(20, 10))
for i, column in enumerate(numeric_columns, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[column])
    plt.title(f'Distribution of {column}')
    plt.tight_layout()

plt.show()

# Count and visualization of categorical variable data
category_columns = df.select_dtypes(include=['object']).columns

plt.figure(figsize=(15, 10))
for i, column in enumerate(category_columns, 1):
    plt.subplot(3, 3, i)
    sns.countplot(
        y=df[column], 
        order=df[column].value_counts().index,
        hue=df[column],
        palette="Set2",
        dodge=False,
        legend=False
    )
    plt.title(f'Count of {column}')
    plt.tight_layout()

plt.show()

No individuals with CALC: Always are present in the training data.

The Obesity\_Type\_III class is slightly overrepresented.

### Test Data Visualization

In [ ]:
df = df_test.copy()

# Visualization of numerical variable distributions via histogram
numeric_columns = df.select_dtypes(include=['float64']).columns

plt.figure(figsize=(20, 10))
for i, column in enumerate(numeric_columns, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[column])
    plt.title(f'Distribution of {column}')
    plt.tight_layout()

plt.show()

# Count and visualization of categorical variable data
category_columns = df.select_dtypes(include=['object']).columns

plt.figure(figsize=(15, 10))
for i, column in enumerate(category_columns, 1):
    plt.subplot(3, 3, i)
    sns.countplot(
        y=df[column], 
        order=df[column].value_counts().index,
        hue=df[column],
        palette="Set2",
        dodge=False,
        legend=False
    )
    plt.title(f'Count of {column}')
    plt.tight_layout()

plt.show()

### Visualization of Relationships with the Target Variable (NObeyesdad)

In [ ]:
df = df_train.copy()

# Define the order of categorical variables
category_order = ['Insufficient_Weight','Normal_Weight',  'Overweight_Level_I', 'Overweight_Level_II', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III']

# Visualization of relationships with categorical data

plt.figure(figsize=(15, 20))

# Gender
plt.subplot(4, 2, 1)
sns.countplot(x='Gender', hue='NObeyesdad', data=df, hue_order=category_order)
plt.title('NObeyesdad Rate by Gender')

#family_history_with_overweight
plt.subplot(4, 2, 2)
sns.countplot(x='family_history_with_overweight', hue='NObeyesdad', data=df, hue_order=category_order)
plt.title('NObeyesdad Rate by family_history_with_overweight')

# FAVC
plt.subplot(4, 2, 3)
sns.countplot(x='FAVC', hue='NObeyesdad', data=df, hue_order=category_order)
plt.title('NObeyesdad Rate by FAVC')

# CAEC
plt.subplot(4, 2, 4)
sns.countplot(x='CAEC', hue='NObeyesdad', data=df, hue_order=category_order)
plt.title('NObeyesdad Rate by CAEC')

#SMOKE
plt.subplot(4, 2, 5)
sns.countplot(x='SMOKE', hue='NObeyesdad', data=df, hue_order=category_order)
plt.title('NObeyesdad Rate by SMOKE')

#SCC
plt.subplot(4, 2, 6)
sns.countplot(x='SCC', hue='NObeyesdad', data=df, hue_order=category_order)
plt.title('NObeyesdad Rate by SCC')

# CALC
plt.subplot(4, 2, 7)
sns.countplot(x='CALC', hue='NObeyesdad', data=df, hue_order=category_order)
plt.title('NObeyesdad Rate by CALC')

# MTRANS
plt.subplot(4, 2, 8)
sns.countplot(x='MTRANS', hue='NObeyesdad', data=df, hue_order=category_order)
plt.title('NObeyesdad Rate by MTRANS')

plt.tight_layout()
plt.show()

In [ ]:
df = df_train.copy()

# Visualization of relationships between numerical variables
plt.figure(figsize=(20, 30))

plt.subplot(8, 1, 1)
sns.histplot(data=df, x='Age', hue='NObeyesdad', bins=30, kde=True, hue_order=category_order)
plt.title('Age')

plt.subplot(8, 1, 2)
sns.histplot(data=df, x='Height', hue='NObeyesdad', bins=30, kde=True, hue_order=category_order)
plt.title('Height')

plt.subplot(8, 1, 3)
sns.histplot(data=df, x='Weight', hue='NObeyesdad', bins=30, kde=True, hue_order=category_order)
plt.title('Weight')

plt.subplot(8, 1, 4)
sns.histplot(data=df, x='FCVC', hue='NObeyesdad', bins=30, kde=True, hue_order=category_order)
plt.title('FCVC')

plt.subplot(8, 1, 5)
sns.histplot(data=df, x='NCP', hue='NObeyesdad', bins=30, kde=True, hue_order=category_order)
plt.title('NCP')

plt.subplot(8, 1, 6)
sns.histplot(data=df, x='CH2O', hue='NObeyesdad', bins=30, kde=True, hue_order=category_order)
plt.title('CH2O')

plt.subplot(8, 1, 7)
sns.histplot(data=df, x='FAF', hue='NObeyesdad', bins=30, kde=True, hue_order=category_order)
plt.title('FAF')

plt.subplot(8, 1, 8)
sns.histplot(data=df, x='TUE', hue='NObeyesdad', bins=30, kde=True, hue_order=category_order)
plt.title('TUE')

plt.tight_layout()
plt.show()

Weight appears to be highly relevant for predicting NObeyesdad, so computing the BMI (Body Mass Index) could be valuable.

### Adding a New Explanatory Variable (BMI)

In [ ]:
# Function to calculate BMI (Body Mass Index)
def calculate_bmi(weight, height):
    height_m = height   
    bmi = weight / (height_m ** 2)
    return bmi

# Add BMI column to the data
df['BMI'] = df.apply(lambda row: calculate_bmi(row['Weight'], row['Height']), axis=1)

# Display the DataFrame for verification
print(df.head())

In [ ]:
# Visualization of relationships with the target variable
plt.figure(figsize=(10, 5))

sns.histplot(data=df, x='BMI', hue='NObeyesdad', bins=30, kde=True, hue_order=category_order)
plt.title('NObeyesdad Rate by BMI')

plt.tight_layout()
plt.show()

Indeed, BMI is highly representative of the NObeyesdad classes.

## Model Training

### Preprocessing: Encoding and Normalization

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

df = df_train.copy()

# Remove the target variable and identifier
X = df.drop(columns=['NObeyesdad', 'id'])

# Codificação das colunas ordinais ou booleanas
le_dict = {}
colonnes_label = [
    'Gender',
    'family_history_with_overweight',
    'FAVC',
    'CAEC',
    'SMOKE',
    'SCC',
    'CALC'
]

for col in colonnes_label:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

# Codificação one-hot para as colunas categóricas não ordinais
X = pd.get_dummies(X, columns=['MTRANS'], drop_first=True)

# Encode the target (dependent variable)
y = df['NObeyesdad']
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

# Normalização com StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    'RandomForest': RandomForestClassifier(),
    "XGBoost": XGBClassifier(),
    'SVM': SVC(),
    'KNN': KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(max_iter=10000, solver='lbfgs'),
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{name} : {scores.mean():.4f} (+/- {scores.std():.4f})")

RandomForest: 0.8985 (+/- 0.0026)

XGBoost: 0.9023 (+/- 0.0049)

SVM: 0.8617 (+/- 0.0064)

KNN: 0.7625 (+/- 0.0052)

Logistic Regression: 0.8603 (+/- 0.0046)

The most effective model is **XGBoost**. Let's try to optimize it.

### Add BMI (Body Mass Index) to the Data

In [ ]:
# Add BMI column to training data
df_with_bmi = df_train.copy()
df_with_bmi['BMI'] = df_with_bmi.apply(lambda row: row['Weight'] / (row['Height'] ** 2), axis=1)

In [ ]:
# Remove the target variable and identifier
X = df_with_bmi.drop(columns=['NObeyesdad', 'id'])

# Codificação das colunas ordinais ou booleanas
le_dict = {}
colonnes_label = [
    'Gender',
    'family_history_with_overweight',
    'FAVC',
    'CAEC',
    'SMOKE',
    'SCC',
    'CALC'
]

for col in colonnes_label:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

# Codificação one-hot para as colunas categóricas não ordinais
X = pd.get_dummies(X, columns=['MTRANS'], drop_first=True)

# Codificação do alvo
y = df_with_bmi['NObeyesdad']
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

# Normalização com StandardScaler
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Model training
model = XGBClassifier(random_state=42)
model.fit(X_train, y_train)
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print(f"XGBoost : {scores.mean():.4f} (+/- {scores.std():.4f})")

XGBoost: 0.9023 (+/- 0.0049)

### Feature Importance for XGBoost

In [ ]:
from xgboost import XGBClassifier, plot_importance
import matplotlib.pyplot as plt

plot_importance(model, max_num_features=10)
plt.tight_layout()
plt.show()


BMI ranks first in the table — this addition is highly relevant.

### Hyperparameter Search to Improve XGBoost Performance

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.7, 1],
    'colsample_bytree': [0.7, 1]
}

grid = GridSearchCV(XGBClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print(f"Best score: {grid.best_score_:.4f}")
print(f"Best params: {grid.best_params_}")


Best score: 0.9070

Best params: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.7}

In [ ]:
best_model = XGBClassifier(
    colsample_bytree=0.7,
    learning_rate=0.1,
    max_depth=5,
    n_estimators=200,
    subsample=0.7,
    random_state=42
)
best_model.fit(X_train, y_train)
scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"XGBoost : {scores.mean():.4f} (+/- {scores.std():.4f})")

Accuracy with the found hyperparameters: 0.9070 (+/- 0.0040)

## Training with 100% of Training Data and Submission File Creation

In [ ]:
# Loading the data
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

# Compute BMI on both datasets
df_train['BMI'] = df_train.apply(lambda row: row['Weight'] / (row['Height'] ** 2), axis=1)
df_test['BMI'] = df_test.apply(lambda row: row['Weight'] / (row['Height'] ** 2), axis=1)

# Variáveis para codificar com LabelEncoder
colonnes_label = [
    'Gender',
    'family_history_with_overweight',
    'FAVC',
    'CAEC',
    'SMOKE',
    'SCC',
    'CALC'
]

# Codificação
le_dict = {}
for col in colonnes_label:
    le = LabelEncoder()
    df_test[col] = le.fit_transform(df_test[col])
    df_train[col] = le.transform(df_train[col])
    le_dict[col] = le

# Codificação do alvo
le_target = LabelEncoder()
y = le_target.fit_transform(df_train['NObeyesdad'])

# Preparation de X
X = df_train.drop(columns=['NObeyesdad', 'id'])
X_test = df_test.drop(columns=['id'])

# Codificação one-hot de MTRANS
X = pd.get_dummies(X, columns=['MTRANS'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['MTRANS'], drop_first=True)

# Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# Retrain the model with the best parameters
final_model = XGBClassifier(
    colsample_bytree=0.7,
    learning_rate=0.1,
    max_depth=5,
    n_estimators=200,
    subsample=0.7
)
final_model.fit(X_scaled, y)

# Predições
y_pred_encoded = final_model.predict(X_test_scaled)
y_pred_labels = le_target.inverse_transform(y_pred_encoded)

# Generate the submission
submission = pd.DataFrame({
    'id': df_test['id_original'] if 'id_original' in df_test else pd.read_csv("test.csv")['id'],
    'NObeyesdad': y_pred_labels
})

submission.to_csv("submission.csv", index=False)


### Submission File Format Verification

In [ ]:
submission.head()
print(submission.columns)

sample = pd.read_csv("sample_submission.csv")
print(submission.columns.equals(sample.columns))
print(submission.shape == sample.shape)    

print(len(submission) == len(df_test))


## Kaggle Results

Accuracy:

Private score: 0.90715

Public score: 0.91257

## Key Results

| Metric | Value |
|--------|-------|
| **Best model** | XGBoost |
| **Public score** | 0.91257 |
| **Private score** | 0.90715 |
| **Most important feature** | BMI (engineered from Height and Weight) |
| **Models compared** | Logistic Regression, KNN, SVM, Random Forest, XGBoost |

| Model | Cross-Validation Accuracy |
|-------|--------------------------|
| Random Forest | 0.8985 (± 0.0026) |
| **XGBoost** | **0.9023 (± 0.0049)** |
| SVM | 0.8617 (± 0.0064) |
| KNN | 0.7625 (± 0.0052) |
| Logistic Regression | 0.8603 (± 0.0046) |

**Key insight:** Engineering the BMI feature from Height and Weight proved to be the single most impactful improvement, becoming the #1 feature by importance in the final model.